# Demo 1: Interactive Geolocation Prediction

Upload a street-view image and let GeoTX predict its geographic location.
The model outputs a probability distribution over 100K candidate locations,
visualized as a heatmap on a Mercator projection world map.

In [1]:
import sys
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ipywidgets import FileUpload, Output, VBox, Label
from IPython.display import display, clear_output

# Add project root and import shared utilities
_root = Path.cwd().parent if Path.cwd().name == 'demos' else Path.cwd()
sys.path.insert(0, str(_root))

from demos.demo_utils import (
    load_geotx_model, load_image, load_gps_gallery, plot_world_heatmap,
)

In [2]:
# Determine device and load model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model = load_geotx_model(device)
gps_gallery = load_gps_gallery().to(device)
print('Model & Weights loaded successfully.')

Using device: cuda


Loading GeoCLIP:  20%|██        | 1/5 [00:16<01:07, 16.99s/step, Build model (CLIP from config, no HF download)]

LoRA applied to CLIP ViT layers 18-23 (r=4, alpha=8, dropout=0.05, targets=['v_proj', 'q_proj'])


Loading GeoCLIP: 100%|██████████| 5/5 [00:25<00:00,  5.08s/step, Done]                                          


GeoTX model loaded on cuda (queue_size=2048, lora_r=4, lora_alpha=8)
Model & Weights loaded successfully.


In [ ]:
# --- File Upload Widget ---
uploader = FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload street-view image'
)
status_label = Label(value='')
output_area = Output()

display(VBox([uploader, status_label, output_area]))

In [4]:
# Re-execute the fixed on_upload function
from pathlib import Path

def on_upload(change):
    with output_area:
        clear_output(wait=True)
        
        uploaded = uploader.value
        if not uploaded:
            return
        
        # Save uploaded image to a temp file
        file_info = uploaded[0]
        fname = file_info['name']
        img_bytes = file_info['content']
        tmp_path = Path('/tmp') / fname
        tmp_path.write_bytes(img_bytes)
        
        status_label.value = 'Image received and preprocessed.'
        
        # Preprocess image
        img_tensor = load_image(tmp_path, device)
        
        # Forward pass: compute logits over GPS gallery
        with torch.no_grad():
            logits = model(img_tensor, model.gps_gallery)
            probs = logits.softmax(dim=-1).cpu().numpy().flatten()
        
        # Top-1 prediction
        top1_idx = int(np.argmax(probs))
        top1_prob = float(probs[top1_idx])
        gallery_np = model.gps_gallery.cpu().numpy()
        pred_lat, pred_lon = gallery_np[top1_idx]
        
        # Display input image
        fig_img, ax_img = plt.subplots(figsize=(5, 5))
        ax_img.imshow(Image.open(tmp_path))
        ax_img.set_title('Uploaded Image', fontsize=11)
        ax_img.axis('off')
        plt.show()
        
        print(f'Top-1 Prediction: ({pred_lat:.4f}, {pred_lon:.4f})  '
              f'(confidence: {top1_prob:.4f})')
        
        # --- Plot heatmap ---
        # Downsample to 20K points for faster rendering
        n_show = 20000
        idx = np.random.RandomState(42).choice(len(probs), n_show, replace=False)
        show_lats = gallery_np[idx, 0]
        show_lons = gallery_np[idx, 1]
        show_vals = probs[idx]
        
        plt.figure(figsize=(14, 7))
        plot_world_heatmap(
            show_lats, show_lons, show_vals,
            title=f'Prediction Heatmap | Top-1: ({pred_lat:.3f}, {pred_lon:.3f})',
            marker_lat=pred_lat, marker_lon=pred_lon,
            marker_label=f'Prediction (p={top1_prob:.4f})',
            alpha=0.5, s=3.0,
        )
        plt.show()
        
        status_label.value = 'Evaluation complete. Prediction plotted.'
        tmp_path.unlink(missing_ok=True)

uploader.observe(on_upload, names='value')
print("on_upload function redefined successfully.")

on_upload function redefined successfully.
